In [28]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# Cargar el dataset preprocesado
df = pd.read_csv("C:/Users/basti/proyecto_cd/data/processed/spotify_dataset_processed.csv", sep=",")

# Definir las características acústicas que usará el modelo
features = ['danceability', 'energy', 'loudness', 'speechiness', 
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# Crear un DataFrame solo con estas características (nuestro "espacio de 9 dimensiones")
X = df[features]

In [29]:
# Inicializar el modelo KNN No Supervisado
# n_neighbors: Queremos 6 vecinos (la canción original + las 5 recomendaciones)
# metric: Usaremos la distancia del coseno
# algorithm: 'brute' o 'auto'. 'brute' calcula todas las distancias, es seguro para este tamaño.
knn_model = NearestNeighbors(n_neighbors=6, metric='cosine', algorithm='brute')

# Ajustar (entrenar) el modelo con nuestro espacio de características
print("Entrenando el modelo espacial KNN...")
knn_model.fit(X)
print("¡Modelo entrenado exitosamente!")

Entrenando el modelo espacial KNN...
¡Modelo entrenado exitosamente!


In [30]:
def buscar_cancion_id(nombre, dataset, artista=None):
    """
    Busca el track_id por nombre exacto y por artista de forma parcial.
    Ej: Si el artista es 'Bizarrap', coincidirá con 'Shakira, Bizarrap'.
    """
    # 1. Filtramos primero por el nombre de la canción (exacto, sin importar mayúsculas)
    filtro_nombre = dataset['track_name'].str.lower() == nombre.lower()
    resultado = dataset[filtro_nombre]
    
    if resultado.empty:
        print(f"❌ No se encontró ninguna canción llamada '{nombre}' en el dataset.")
        return None
    
    # 2. Si el usuario ingresó un artista, aplicamos el filtro parcial
    if artista:
        # Usamos contains para coincidencia parcial
        filtro_artista = resultado['artists'].str.contains(artista, case=False, na=False)
        resultado = resultado[filtro_artista]
        
        if resultado.empty:
            print(f"❌ Se encontró la canción '{nombre}', pero NO asociada al artista '{artista}'.")
            return None
            
    # 3. Si hay más de un resultado (ej. remasterizaciones o duplicados no filtrados)
    if len(resultado) > 1:
        artista_encontrado = resultado.iloc[0]['artists']
        print(f"⚠️ Aviso: Hay múltiples versiones. Usando la primera: {nombre} - {artista_encontrado}")
        
    return resultado.iloc[0]['track_id']

In [45]:
import time
import sys

# ==========================================
# 1. SOLUCIÓN AL IMPORT DEL EVALUADOR
# ==========================================
# Le decimos a Python que busque en la carpeta 'src' que está un nivel más atrás.
# Si tu notebook está en 'proyecto_cd/notebooks', esto encontrará 'proyecto_cd/src'
sys.path.append('../src')

# Si la ruta relativa falla, descomenta la siguiente línea y usa tu ruta absoluta:
# sys.path.append('C:/Users/basti/proyecto_cd/src')

try:
    from evaluador import evaluar_modelo_recomendacion
    print("✅ Módulo 'evaluador.py' importado correctamente.")
except ImportError:
    print("❌ Error crítico: No se pudo encontrar 'evaluador.py'. Revisa la ruta.")


# ==========================================
# 2. SOLUCIÓN AL NAME ERROR (Función faltante)
# ==========================================
def recomendar_por_nombre_knn_y_evaluar(nombre_cancion, modelo, dataframe, features_list, artista=None, top_k=5):
    """
    Busca la canción por nombre, ejecuta el modelo KNN, genera la explicabilidad,
    y auto-evalúa su latencia total y distancia acústica.
    """
    # ⏱️ 1. INICIAR CRONÓMETRO (Latencia percibida por el usuario)
    inicio_tiempo = time.time()
    
    # 2. Buscar el ID de la canción
    track_id = buscar_cancion_id(nombre_cancion, dataframe, artista)
    
    if track_id:
        # Extraemos el vector semilla para enviarlo al evaluador
        vector_semilla = dataframe[dataframe['track_id'] == track_id][features_list]
        
        # 3. Ejecutar el motor base del KNN (El que calcula y explica)
        recomendaciones_texto, vectores_recomendados = recomendar_canciones_knn(
            track_id_semilla=track_id, 
            modelo=modelo, 
            dataframe=dataframe, 
            features_list=features_list, 
            top_k=top_k
        )
        
        # ⏱️ 4. DETENER CRONÓMETRO
        fin_tiempo = time.time()
        
        # 5. AUTO-EVALUACIÓN: Llamamos al evaluador
        if vectores_recomendados is not None:
            try:
                evaluar_modelo_recomendacion(
                    nombre_modelo="K-Nearest Neighbors (KNN)",
                    vector_semilla=vector_semilla,
                    vectores_recomendados=vectores_recomendados,
                    tiempo_inicio=inicio_tiempo,
                    tiempo_fin=fin_tiempo
                )
            except NameError:
                print("⚠️ No se mostraron los KPIs porque el evaluador no cargó correctamente.")
                
        return recomendaciones_texto, vectores_recomendados
    
    return None, None

✅ Módulo 'evaluador.py' importado correctamente.


In [42]:
def iniciar_app_recomendacion(df, knn_model, features_list):
    """
    Inicia una interfaz interactiva por consola para probar los modelos en vivo.
    """
    print("=" * 65)
    print("🎧 BIENVENIDO AL SISTEMA DE RECOMENDACIÓN MUSICAL (CRISP-DM) 🎧")
    print("=" * 65)
    
    while True:
        print("\n" + "-" * 65)
        print("(Escribe 'salir' en el nombre de la canción para terminar la prueba)")
        
        # Recibir entradas del usuario
        entrada_cancion = input("🎵 Ingresa el nombre de la canción: ").strip()
        
        if entrada_cancion.lower() == 'salir':
            print("\n¡Gracias por probar el sistema! Cerrando... 👋")
            break
            
        entrada_artista = input("🎤 Ingresa el artista (o presiona Enter para omitir): ").strip()
        
        # Si el usuario solo presiona Enter, lo dejamos como None
        if entrada_artista == "":
            entrada_artista = None
            
        print("\nProcesando solicitud...")
        
        # 1. Ejecutar la búsqueda universal
        track_id_encontrado = buscar_cancion_id(entrada_cancion, df, entrada_artista)
        
        if track_id_encontrado:
            print("=" * 65)
            print("🚀 RESULTADOS DE LA BÚSQUEDA Y EVALUACIÓN DE KPIs 🚀")
            print("=" * 65)
            
           # --- EJECUTAR KNN AUTO-EVALUADO ---
            print("\n" + "*" * 20 + " MODELO KNN " + "*" * 20)
            recomendar_por_nombre_knn_y_evaluar(
                nombre_cancion=entrada_cancion, 
                modelo=knn_model, 
                dataframe=df, 
                features_list=features_list, 
                artista=entrada_artista
            )
            # -------------------------------------------------------------
            # NOTA PARA EL NOTEBOOK DE COMPARACIÓN:
            # Si estás en el notebook donde tienen los 3 modelos integrados, 
            # aquí puedes descomentar las líneas de abajo para correr todos a la vez.
            # -------------------------------------------------------------
            
            # print("\n" + "*" * 20 + " MODELO K-MEANS " + "*" * 20)
            # probar_kmeans_por_nombre(entrada_cancion, df, features_list, artista=entrada_artista)
            
            # print("\n" + "*" * 20 + " MODELO DBSCAN " + "*" * 20)
            # probar_dbscan_por_nombre(entrada_cancion, df, features_list, artista=entrada_artista)

In [43]:
import numpy as np
import pandas as pd
import time
import sys

def generar_explicacion_acustica(vector_semilla, vector_recomendado, features_list):
    """
    Compara dos vectores acústicos y identifica las 3 variables con menor diferencia
    para justificar la recomendación al usuario.
    """
    # Calculamos la diferencia absoluta por cada característica
    diferencias = np.abs(vector_semilla.values[0] - vector_recomendado.values[0])
    
    # Mapeo de nombres técnicos a nombres amigables para el usuario
    traduccion = {
        'danceability': 'Bailabilidad',
        'energy': 'Nivel de Energía',
        'loudness': 'Volumen (Loudness)',
        'speechiness': 'Presencia Vocal',
        'acousticness': 'Acústica',
        'instrumentalness': 'Instrumentalidad',
        'liveness': 'Sensación de Vivo',
        'valence': 'Positividad Musical',
        'tempo': 'Ritmo (Tempo)'
    }
    
    # Crear diccionario de diferencias y ordenar
    dif_dict = {features_list[i]: diferencias[i] for i in range(len(features_list))}
    top_3 = sorted(dif_dict.items(), key=lambda item: item[1])[:3]
    
    # Formatear la explicación
    vars_explicacion = [traduccion[var[0]] for var in top_3]
    return f"💡 Recomendada por su similitud en: {vars_explicacion[0]}, {vars_explicacion[1]} y {vars_explicacion[2]}."

# Asegurarnos de tener el evaluador importado
try:
    from evaluador import evaluar_modelo_recomendacion
except ImportError:
    print("⚠️ Recuerda importar evaluar_modelo_recomendacion desde src/evaluador.py")

def recomendar_canciones_knn(track_id_semilla, modelo, dataframe, features_list, top_k=5):
    """
    Motor de recomendación puro. Busca los vecinos y genera explicaciones.
    NO evalúa métricas, solo retorna los datos para que sean evaluados superiormente.
    """
    try:
        idx_semilla = dataframe.index[dataframe['track_id'] == track_id_semilla].tolist()[0]
    except IndexError:
        print(f"❌ Error: El ID '{track_id_semilla}' no existe en el dataset.")
        return None, None

    vector_semilla = dataframe.iloc[[idx_semilla]][features_list]
    nombre_semilla = dataframe.iloc[idx_semilla]['track_name']
    artista_semilla = dataframe.iloc[idx_semilla]['artists']
    
    print(f"🔍 Generando recomendaciones para: '{nombre_semilla}' de {artista_semilla}")
    print("-" * 70)

    distancias, indices = modelo.kneighbors(vector_semilla, n_neighbors=top_k + 1)

    lista_recomendaciones = []
    indices_recomendados = []
    
    for i in range(1, len(distancias[0])):
        idx_vecino = indices[0][i]
        distancia_matematica = distancias[0][i]
        
        track_rec = dataframe.iloc[idx_vecino]
        vector_rec = dataframe.iloc[[idx_vecino]][features_list]
        
        explicacion = generar_explicacion_acustica(vector_semilla, vector_rec, features_list)
        
        print(f"{i}. 🎵 {track_rec['track_name']} - {track_rec['artists']}")
        print(f"   {explicacion}")
        print(f"   (Distancia del Coseno: {distancia_matematica:.4f})\n")
        
        lista_recomendaciones.append({
            'track_name': track_rec['track_name'],
            'artists': track_rec['artists'],
            'distancia': distancia_matematica,
            'explicacion': explicacion
        })
        
        indices_recomendados.append(idx_vecino)
        
    print("-" * 70)
    
    # Retornar los vectores acústicos puros
    vectores_df = dataframe.iloc[indices_recomendados][features_list]
    
    return lista_recomendaciones, vectores_df

⚠️ Recuerda importar evaluar_modelo_recomendacion desde src/evaluador.py


In [46]:
# Iniciar la interfaz de prueba en vivo
iniciar_app_recomendacion(df, knn_model, features)

🎧 BIENVENIDO AL SISTEMA DE RECOMENDACIÓN MUSICAL (CRISP-DM) 🎧

-----------------------------------------------------------------
(Escribe 'salir' en el nombre de la canción para terminar la prueba)


🎵 Ingresa el nombre de la canción:  Stay
🎤 Ingresa el artista (o presiona Enter para omitir):  



Procesando solicitud...
⚠️ Aviso: Hay múltiples versiones. Usando la primera: Stay - Cradle Of Filth
🚀 RESULTADOS DE LA BÚSQUEDA Y EVALUACIÓN DE KPIs 🚀

******************** MODELO KNN ********************
⚠️ Aviso: Hay múltiples versiones. Usando la primera: Stay - Cradle Of Filth
🔍 Generando recomendaciones para: 'Stay' de Cradle Of Filth
----------------------------------------------------------------------
1. 🎵 終究還是因為愛 - TRASH
   💡 Recomendada por su similitud en: Positividad Musical, Acústica y Sensación de Vivo.
   (Distancia del Coseno: 0.0003)

2. 🎵 Dear Agony - Aurora Version - Breaking Benjamin;Lacey Sturm
   💡 Recomendada por su similitud en: Instrumentalidad, Acústica y Presencia Vocal.
   (Distancia del Coseno: 0.0006)

3. 🎵 Zor - maNga
   💡 Recomendada por su similitud en: Acústica, Presencia Vocal y Sensación de Vivo.
   (Distancia del Coseno: 0.0007)

4. 🎵 Robot Boy - Linkin Park
   💡 Recomendada por su similitud en: Acústica, Instrumentalidad y Nivel de Energía.
   (D

🎵 Ingresa el nombre de la canción:  salir



¡Gracias por probar el sistema! Cerrando... 👋
